#Circuits Data ingestion 

Import env variables

In [0]:
%run ../00-common/01-env-variables

Import helper functions

In [0]:
%run ../00-common/02-bronze-helpers

In [0]:
dbutils.widgets.text('batch_id' , "")
v_batch_id = dbutils.widgets.get('batch_id')

In [0]:
source = f"{landing_file_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StringType, DoubleType, StructField, StructType

circuits_schema = StructType([
    StructField('circuitId' , StringType()),
    StructField('url' , StringType()),
    StructField('circuitName' , StringType()),
    StructField('lat' , DoubleType()),
    StructField('lng' , DoubleType()),
    StructField('locality' , StringType()),
    StructField('country' , StringType()),
])

In [0]:
circuits_df = (
    spark.read.format('csv')
    .option('header', True)
   # .option('inferSchema', True)
    .schema(circuits_schema)
    .load(source)
)

In [0]:
display(circuits_df)

## Adding metadata

In [0]:
from pyspark.sql import functions as F
circuits_df = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_df)

In [0]:
# circuits_final_df = circuits_df.withColumn('batch_id' , F.lit(v_batch_id))
# (
#     circuits_final_df.
#     write
#     .format('delta')
#     .mode('overwrite')
#     .partitionBy('batch_id')
#     .option('replaceWhere' , f"batch_id = '{v_batch_id}'")
#     .saveAsTable(table_name)
# )

write_to_bronze(circuits_final_df,table_name,v_batch_id)

In [0]:
%sql
select * from formula1incr.bronze.circuits
